In [ ]:
%matplotlib inline
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scanpy.external as sce
import matplotlib.pyplot as plt
import singlecellexperiment as scet
import seaborn as sns
from matplotlib.patches import Circle
import scimap as sm
from matplotlib.patches import Patch
import airr
import glob

In [ ]:
import warnings
warnings.filterwarnings('ignore')

### Function

In [ ]:
# ---- Merge-report helpers (adapted from the scBCR/scIntegrator Quarto merge report) ----
try:
    from itables import show as show_itables
    import itables.options as itables_options
    HAS_ITABLES = True
    itables_options.lengthMenu = [10, 25, 50, 100]
    itables_options.pageLength = 5
    itables_options.scrollX = True
except ImportError:
    HAS_ITABLES = False

from IPython.display import display, Markdown

def show_subheader(text):
    display(Markdown(f'### {text}'))

def show_table(df, page_length=25):
    if df is None:
        return
    if HAS_ITABLES:
        rendered = show_itables(
            df, paging=True, pageLength=page_length,
            lengthChange=True, scrollX=True, classes='display compact nowrap',
        )
        if rendered is not None:
            display(rendered)
    else:
        display(df)

def pick_first_existing(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None

def summarize_counts(df, dataset_label, group_col, cell_col, out_col):
    if group_col is None:
        return None
    x = df.copy()
    x[group_col] = x[group_col].astype(str)
    if cell_col is not None and cell_col in x.columns:
        x[cell_col] = x[cell_col].astype(str)
    out = (
        x.groupby(group_col, dropna=False)
         .agg(
             n_cells=(cell_col, pd.Series.nunique) if cell_col else (group_col, 'size'),
             n_sequences=(group_col, 'size')
         )
         .reset_index()
         .rename(columns={group_col: out_col})
         .sort_values(out_col)
         .reset_index(drop=True)
    )
    out.insert(0, 'dataset', dataset_label)
    return out

def summarize_cells_only(df, dataset_label, group_col, cell_col, out_col):
    if group_col is None:
        return None
    x = df.copy()
    x[group_col] = x[group_col].astype(str)
    x[cell_col] = x[cell_col].astype(str)
    out = (
        x.groupby(group_col, dropna=False)
         .agg(n_cells=(cell_col, pd.Series.nunique))
         .reset_index()
         .rename(columns={group_col: out_col})
         .sort_values(out_col)
         .reset_index(drop=True)
    )
    out.insert(0, 'dataset', dataset_label)
    return out

def plot_counts(df, category_col, title, metrics=('n_cells', 'n_sequences')):
    if df is None or df.empty:
        show_subheader(f'{title}: no data available')
        return
    value_vars = [m for m in metrics if m in df.columns]
    if not value_vars:
        show_subheader(f'{title}: no plottable metrics')
        return
    long_df = df.melt(id_vars=[category_col], value_vars=value_vars, var_name='metric', value_name='count')
    plt.figure(figsize=(max(8, len(df) * 0.8), 4.8))
    sns.barplot(data=long_df, x=category_col, y='count', hue='metric')
    plt.title(title)
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Metric', bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)
    plt.tight_layout(rect=[0, 0, 0.84, 1])
    plt.show()

def summarize_chain_distribution(df, group_col, chain_col):
    if group_col is None or chain_col is None or group_col not in df.columns or chain_col not in df.columns:
        return None
    x = df.copy()
    x[group_col] = x[group_col].astype(str)
    x[chain_col] = x[chain_col].fillna('NA').astype(str)
    out = (
        x.groupby([group_col, chain_col], dropna=False)
         .size()
         .reset_index(name='n_sequences')
         .rename(columns={group_col: 'group_id', chain_col: 'chain'})
         .sort_values(['group_id', 'chain'])
         .reset_index(drop=True)
    )
    return out

def plot_chain_distribution(chain_df, title):
    if chain_df is None or chain_df.empty:
        show_subheader(f'{title}: chain column unavailable')
        return
    pivot = chain_df.pivot(index='group_id', columns='chain', values='n_sequences').fillna(0)
    ax = pivot.plot(kind='bar', stacked=False, figsize=(max(9, len(pivot) * 0.8), 5))
    plt.title(title)
    plt.xlabel('sample_id')
    plt.ylabel('Number of sequences')
    plt.xticks(rotation=45, ha='right')
    ax.legend(title='Chain', bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)
    plt.tight_layout(rect=[0, 0, 0.82, 1])
    plt.show()

def plot_bcr_celltypes(counts, title, normalize=False):
    data = counts.div(counts.sum(axis=1), axis=0) * 100 if normalize else counts
    ylabel = '% of BCR+ cells' if normalize else 'Number of BCR+ cells'

    fig, ax = plt.subplots(figsize=(max(6, 0.7 * len(data)), 5))
    data.plot(kind='bar', stacked=True, ax=ax, colormap='tab20', edgecolor='white', width=0.8)

    ax.set_xlabel(sample_col)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(title='cell type', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()# ---- Merge-report helpers (adapted from the scBCR/scIntegrator Quarto merge report) ----
try:
    from itables import show as show_itables
    import itables.options as itables_options
    HAS_ITABLES = True
    itables_options.lengthMenu = [10, 25, 50, 100]
    itables_options.pageLength = 5
    itables_options.scrollX = True
except ImportError:
    HAS_ITABLES = False

from IPython.display import display, Markdown

def show_subheader(text):
    display(Markdown(f'### {text}'))

def show_table(df, page_length=25):
    if df is None:
        return
    if HAS_ITABLES:
        rendered = show_itables(
            df, paging=True, pageLength=page_length,
            lengthChange=True, scrollX=True, classes='display compact nowrap',
        )
        if rendered is not None:
            display(rendered)
    else:
        display(df)

def pick_first_existing(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None

def summarize_counts(df, dataset_label, group_col, cell_col, out_col):
    if group_col is None:
        return None
    x = df.copy()
    x[group_col] = x[group_col].astype(str)
    if cell_col is not None and cell_col in x.columns:
        x[cell_col] = x[cell_col].astype(str)
    out = (
        x.groupby(group_col, dropna=False)
         .agg(
             n_cells=(cell_col, pd.Series.nunique) if cell_col else (group_col, 'size'),
             n_sequences=(group_col, 'size')
         )
         .reset_index()
         .rename(columns={group_col: out_col})
         .sort_values(out_col)
         .reset_index(drop=True)
    )
    out.insert(0, 'dataset', dataset_label)
    return out

def summarize_cells_only(df, dataset_label, group_col, cell_col, out_col):
    if group_col is None:
        return None
    x = df.copy()
    x[group_col] = x[group_col].astype(str)
    x[cell_col] = x[cell_col].astype(str)
    out = (
        x.groupby(group_col, dropna=False)
         .agg(n_cells=(cell_col, pd.Series.nunique))
         .reset_index()
         .rename(columns={group_col: out_col})
         .sort_values(out_col)
         .reset_index(drop=True)
    )
    out.insert(0, 'dataset', dataset_label)
    return out

def plot_counts(df, category_col, title, metrics=('n_cells', 'n_sequences')):
    if df is None or df.empty:
        show_subheader(f'{title}: no data available')
        return
    value_vars = [m for m in metrics if m in df.columns]
    if not value_vars:
        show_subheader(f'{title}: no plottable metrics')
        return
    long_df = df.melt(id_vars=[category_col], value_vars=value_vars, var_name='metric', value_name='count')
    plt.figure(figsize=(max(8, len(df) * 0.8), 4.8))
    sns.barplot(data=long_df, x=category_col, y='count', hue='metric')
    plt.title(title)
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Metric', bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)
    plt.tight_layout(rect=[0, 0, 0.84, 1])
    plt.show()

def summarize_chain_distribution(df, group_col, chain_col):
    if group_col is None or chain_col is None or group_col not in df.columns or chain_col not in df.columns:
        return None
    x = df.copy()
    x[group_col] = x[group_col].astype(str)
    x[chain_col] = x[chain_col].fillna('NA').astype(str)
    out = (
        x.groupby([group_col, chain_col], dropna=False)
         .size()
         .reset_index(name='n_sequences')
         .rename(columns={group_col: 'group_id', chain_col: 'chain'})
         .sort_values(['group_id', 'chain'])
         .reset_index(drop=True)
    )
    return out

def plot_chain_distribution(chain_df, title):
    if chain_df is None or chain_df.empty:
        show_subheader(f'{title}: chain column unavailable')
        return
    pivot = chain_df.pivot(index='group_id', columns='chain', values='n_sequences').fillna(0)
    ax = pivot.plot(kind='bar', stacked=False, figsize=(max(9, len(pivot) * 0.8), 5))
    plt.title(title)
    plt.xlabel('sample_id')
    plt.ylabel('Number of sequences')
    plt.xticks(rotation=45, ha='right')
    ax.legend(title='Chain', bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)
    plt.tight_layout(rect=[0, 0, 0.82, 1])
    plt.show()

def plot_bcr_celltypes(counts, title, normalize=False):
    data = counts.div(counts.sum(axis=1), axis=0) * 100 if normalize else counts
    ylabel = '% of BCR+ cells' if normalize else 'Number of BCR+ cells'

    fig, ax = plt.subplots(figsize=(max(6, 0.7 * len(data)), 5))
    data.plot(kind='bar', stacked=True, ax=ax, colormap='tab20', edgecolor='white', width=0.8)

    ax.set_xlabel(sample_col)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(title='cell type', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

### scRNA and scBCR Data loading and preprocessing

In [ ]:
adata = sc.read_h5ad('/gpfs/gibbs/pi/kleinstein/huimin/D5_all_samples_results/runs/scIntegrator_BN46/results/scanpy_cluster/concat.h5ad')
scbcr_df = pd.read_csv('/gpfs/gibbs/pi/kleinstein/huimin/D5_all_samples_results/runs/scBCR_BN46/results/clonal_analysis/repertoire_analysis/repertoire_analysis_report/repertoires/All_samples__repertoire-pass.tsv', sep='\t')

In [ ]:
adata.obs['sample'] = (
    adata.obs['sample']
    .astype(str)
    .str.replace(r'^filtered_feature_bc_matrix_', '', regex=True)
    .astype('category')
)

In [ ]:
#Collapse fine cell_type -> major cell type and add col 'major_celltype' to the adata
b_cell_set = {
    'Age-associated B cells', 'B cells', 'Cycling B cells', 'Follicular B cells',
    'Germinal center B cells', 'Large pre-B cells', 'Memory B cells', 'Naive B cells',
    'Plasma cells', 'Plasmablasts', 'Pre-pro-B cells', 'Pro-B cells',
    'Proliferative germinal center B cells', 'Small pre-B cells', 'Transitional B cells',
}

t_cell_set = {
    'CD8a/a', 'CD8a/b(entry)', 'CRTAM+ gamma-delta T cells', 'Cycling T cells',
    'Cycling gamma-delta T cells', 'Double-negative thymocytes', 'Double-positive thymocytes',
    'Follicular helper T cells', 'MAIT cells', 'Memory CD4+ cytotoxic T cells',
    'Regulatory T cells', 'T(agonist)', 'Tcm/Naive cytotoxic T cells',
    'Tcm/Naive helper T cells', 'Tem/Effector helper T cells',
    'Tem/Effector helper T cells PD1+', 'Tem/Temra cytotoxic T cells',
    'Tem/Trm cytotoxic T cells', 'Treg(diff)', 'Trm cytotoxic T cells',
    'Type 1 helper T cells', 'Type 17 helper T cells', 'gamma-delta T cells',
    'NKT cells',                       # see note — TCR+, classed as T here
}

nk_cell_set = {
    'CD16+ NK cells', 'CD16- NK cells', 'Cycling NK cells', 'NK cells', 'Transitional NK',
    'ILC', 'ILC precursor', 'ILC1', 'ILC2', 'ILC3',   # innate lymphoid, folded into NK
}

dc_set = {
    'Cycling DCs', 'DC', 'DC precursor', 'DC1', 'DC2', 'DC3',
    'Migratory DCs', 'Transitional DC', 'pDC', 'pDC precursor',
}

def to_high_level(ct):
    ct = str(ct)
    if ct in b_cell_set:  return 'B cells'
    if ct in t_cell_set:  return 'T cells'
    if ct in nk_cell_set: return 'NK cells'
    if ct in dc_set:      return 'DCs'
    return 'Other'        # monocytes, macrophages, erythroid, progenitors, stroma, etc.

adata.obs['major_celltype'] = adata.obs['cell_type'].map(to_high_level)

In [ ]:
b_cell_types = b_cell_set
# Subset B cells
bdata = adata[adata.obs['cell_type'].isin(b_cell_types)].copy()
bdata.obs['cell_type'] = bdata.obs['cell_type'].cat.remove_unused_categories()

In [ ]:
# sample name taken from adata obs_names suffix, e.g. "...-filtered_feature_bc_matrix_<sample>"
adata.obs['sample_from_name'] = adata.obs_names.str.extract(
    r'-filtered_feature_bc_matrix_(.*)$', expand=False
)

adata.obs['key'] = adata.obs['sample_from_name'].astype(str) + '|' + adata.obs['barcode']
scbcr_df['key'] = scbcr_df['sample_id'].astype(str) + '|' + scbcr_df['cell_id_cleaned']

lookup = adata.obs.drop_duplicates('key').set_index('key')

for col in ['cell_type', 'major_celltype']:
    scbcr_df[col] = scbcr_df['key'].map(lookup[col]).astype(object)
    scbcr_df[col] = scbcr_df[col].where(scbcr_df[col].notna(), 'None')

In [ ]:
scbcr_df['bcell'] = np.where(scbcr_df['cell_type'].isin(b_cell_set), 'yes', 'no')

### scRNA data overview

In [ ]:
# --- scIntegrator cells by sample/subject (reuses `adata`, no reload) ---
scIntegrator_sample_col   = pick_first_existing(adata.obs, ['sample', 'sample_id'])
scIntegrator_subject_col  = pick_first_existing(adata.obs, ['subject', 'subject_id'])
scIntegrator_celltype_col = pick_first_existing(adata.obs, ['cell_type', 'celltype', 'annotation', 'cluster'])

scIntegrator_df = pd.DataFrame(index=adata.obs_names).reset_index().rename(columns={'index': 'cell_id_scIntegrator'})
scIntegrator_df['sample_id'] = adata.obs[scIntegrator_sample_col].astype(str).values if scIntegrator_sample_col else 'NA'
if scIntegrator_subject_col:
    scIntegrator_df['subject_id'] = adata.obs[scIntegrator_subject_col].astype(str).values
scIntegrator_df['cell_type'] = adata.obs[scIntegrator_celltype_col].astype(str).values if scIntegrator_celltype_col else 'NA'

scIntegrator_by_sample = (
    summarize_cells_only(scIntegrator_df, 'scIntegrator', 'sample_id', 'cell_id_scIntegrator', 'sample_id')
    if scIntegrator_sample_col else None
)
scIntegrator_by_subject = (
    summarize_cells_only(scIntegrator_df, 'scIntegrator', 'subject_id', 'cell_id_scIntegrator', 'subject_id')
    if 'subject_id' in scIntegrator_df.columns else None
)

show_subheader('Detected scIntegrator column mapping')
show_table(pd.DataFrame([
    {'field': 'sample',    'column_used': scIntegrator_sample_col or '(not available)'},
    {'field': 'subject',   'column_used': scIntegrator_subject_col or '(not available)'},
    {'field': 'cell_type', 'column_used': scIntegrator_celltype_col or '(fallback NA)'},
]))

show_subheader('scIntegrator cells by sample')
show_table(scIntegrator_by_sample)
plot_counts(scIntegrator_by_sample, 'sample_id', 'scIntegrator Cells per Sample', metrics=('n_cells',))

### Identification of BCR+ cell

In [ ]:
# --- 1. get the barcode column from the repertoire (AIRR uses 'cell_id') ---
scbcr_df['cell_id_cleaned'] = scbcr_df['cell_id'].str.split('-').str[:-1].str.join('-')
bcr_cells = scbcr_df['cell_id_cleaned'].astype(str).unique()

adata.obs['barcode'] = adata.obs_names.str.replace(
    r'-filtered_feature_bc_matrix_.*$', '', regex=True
)
bcr_cells = set(scbcr_df['cell_id_cleaned'].astype(str))

adata.obs['in_bcr'] = adata.obs['barcode'].isin(bcr_cells)

# --- 2. match against adata barcodes ---
n_match = adata.obs['in_bcr'].sum()
print(f"BCR barcodes:              {len(bcr_cells)}")
print(f"adata cells matched:       {n_match} ({n_match/adata.n_obs:.1%} of adata)")
print(f"unique adata barcodes hit: {adata.obs.loc[adata.obs['in_bcr'],'barcode'].nunique()}")


# --- 3. subset to BCR+ cells ---
bcr_adata = adata[adata.obs['in_bcr']].copy()

In [ ]:
# --- column setup ---
sample_col = pick_first_existing(adata.obs, ['sample_id', 'sample'])
celltype_col = 'cell_type'

# Adjust this pattern to match your B cell labels (e.g. 'B cell', 'B cells', 'naive B')
bcell_pattern = r'\bB[\s_-]?cell'
adata.obs['is_bcell'] = (
    adata.obs[celltype_col].astype(str).str.contains(bcell_pattern, case=False, regex=True)
)

# --- per-sample counts ---
obs = adata.obs
bcr_counts = pd.DataFrame({
    'BCR+ B cell':     obs[obs['in_bcr'] & obs['is_bcell']].groupby(sample_col, observed=True).size(),
    'BCR+ other type': obs[obs['in_bcr'] & ~obs['is_bcell']].groupby(sample_col, observed=True).size(),
    'BCR- B cell':     obs[~obs['in_bcr'] & obs['is_bcell']].groupby(sample_col, observed=True).size(),
    'BCR- other type': obs[~obs['in_bcr'] & ~obs['is_bcell']].groupby(sample_col, observed=True).size(),
}).fillna(0).astype(int)

bcr_counts['total'] = bcr_counts.sum(axis=1)
bcr_counts['BCR+ total'] = bcr_counts['BCR+ B cell'] + bcr_counts['BCR+ other type']
bcr_counts['B cell total'] = bcr_counts['BCR+ B cell'] + bcr_counts['BCR- B cell']
bcr_counts['% BCR+'] = (bcr_counts['BCR+ total'] / bcr_counts['total'] * 100).round(1)
bcr_counts['% BCR+ that are B cell'] = (
    bcr_counts['BCR+ B cell'] / bcr_counts['BCR+ total'].replace(0, pd.NA) * 100
).astype(float).round(1)
bcr_counts['% B cells with BCR'] = (
    bcr_counts['BCR+ B cell'] / bcr_counts['B cell total'].replace(0, pd.NA) * 100
).astype(float).round(1)

show_subheader('BCR+ / BCR- cells per sample')
show_table(bcr_counts.reset_index())

# --- plot ---
bcr_color, neg_color = '#1f77b4', '#d3d3d3'
samples = bcr_counts.index.astype(str)
x = range(len(samples))

fig, ax = plt.subplots(figsize=(max(6, 0.7 * len(samples)), 5))

b1 = bcr_counts['BCR+ B cell']
b2 = bcr_counts['BCR+ other type']
b3 = bcr_counts['BCR- B cell']
b4 = bcr_counts['BCR- other type']

ax.bar(x, b1, color=bcr_color, hatch='////', edgecolor='white', width=0.8)
ax.bar(x, b2, bottom=b1, color=bcr_color, edgecolor='white', width=0.8)
ax.bar(x, b3, bottom=b1 + b2, color=neg_color, hatch='////', edgecolor='white', width=0.8)
ax.bar(x, b4, bottom=b1 + b2 + b3, color=neg_color, edgecolor='white', width=0.8)

# label % BCR+ on top of each bar
for i, (tot, pct) in enumerate(zip(bcr_counts['total'], bcr_counts['% BCR+'])):
    ax.text(i, tot, f'{pct}%', ha='center', va='bottom', fontsize=8)

ax.set_xticks(list(x))
ax.set_xticklabels(samples, rotation=45, ha='right')
ax.set_xlabel(sample_col)
ax.set_ylabel('Number of cells')
ax.set_title('BCR+ vs BCR- Cells per Sample')

ax.legend(handles=[
    Patch(facecolor=bcr_color, hatch='////', edgecolor='white', label='BCR+ (annotated B cell)'),
    Patch(facecolor=bcr_color, edgecolor='white', label='BCR+ (other cell type)'),
    Patch(facecolor=neg_color, hatch='////', edgecolor='white', label='BCR- (annotated B cell)'),
    Patch(facecolor=neg_color, edgecolor='white', label='BCR- (other cell type)'),
], bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# --- major cell type distribution among BCR+ cells ---
bcr_pos = adata.obs[adata.obs['in_bcr']]

ct_counts = pd.crosstab(bcr_pos[sample_col], bcr_pos['major_celltype'].astype(str))
ct_counts = ct_counts[ct_counts.sum().sort_values(ascending=False).index]  # most common at bottom

show_subheader('Major cell type distribution of BCR+ cells per sample')
show_table(ct_counts.reset_index())

plot_bcr_celltypes(ct_counts, 'Major Cell Type Distribution of BCR+ Cells per Sample')
plot_bcr_celltypes(ct_counts, 'Major Cell Type Distribution of BCR+ Cells per Sample (%)', normalize=True)

### B cell subtype marker expression

In [ ]:
# --- markers (deduped) ---
marker_genes = {
    'B cells':  ['MS4A1', 'CD19', 'IGHM', 'IGHD', 'JCHAIN'],
    'T cells':  ['CD3E', 'CD2', 'CD3D'],
    'NK cells': ['GNLY', 'NKG7'],
    'DCs':      ['CST3', 'FCER1A'],
}
marker_genes = {grp: [g for g in genes if g in adata.var_names]
                for grp, genes in marker_genes.items()}
for grp, genes in marker_genes.items():
    print(f"{grp:10s}: {genes}")

# --- rows: lineage × BCR status ---
lineages = ['B cells', 'T cells', 'NK cells', 'DCs']
sub = adata[adata.obs['major_celltype'].isin(lineages)].copy()
bcr_status = sub.obs['in_bcr'].map({True: 'with BCR', False: 'without BCR'})
sub.obs['group'] = (sub.obs['major_celltype'].astype(str) + ' ' + bcr_status).astype('category')
row_order = [f'{lin} {s}' for lin in lineages for s in ['with BCR', 'without BCR']]
row_order = [r for r in row_order if r in sub.obs['group'].cat.categories]
sub.obs['group'] = sub.obs['group'].cat.reorder_categories(row_order)


def add_brackets(axes_dict, marker_dict, y=1, tick=0.04, fontsize=12):
    """Draw gene-group brackets manually on the dotplot's main axis.
    Robust to the scanpy/matplotlib clipping quirk that hides the built-in ones."""
    ax = axes_dict['mainplot_ax']
    flat = [g for gs in marker_dict.values() for g in gs]
    xpos = {g: i for i, g in enumerate(flat)}      # columns sit at 0,1,2,... centers
    trans = ax.get_xaxis_transform()               # x = data coords, y = axes fraction
    for label, gs in marker_dict.items():
        if not gs:
            continue
        idx = [xpos[g] for g in gs]
        l, r = min(idx)+0.5, max(idx) + 0.5
        ax.plot([l, l, r, r], [y, y + tick, y + tick, y],
                transform=trans, color='black', lw=1.3, clip_on=False)
        ax.text((l + r) / 2, y + tick + 0.01, label, transform=trans,
                ha='center', va='bottom', fontsize=fontsize, clip_on=False)


# --- dotplot; capture the axes dict ---
axd = sc.pl.dotplot(
    sub,
    var_names=marker_genes,
    groupby='group',
    categories_order=row_order,
    standard_scale='var',
    cmap='Blues',
    dendrogram=False,
    figsize=(0.55 * sum(len(g) for g in marker_genes.values()) + 3, 4.5),
    show=False,
)

# hide scanpy's own (clipped) bracket axis, then draw ours
if 'gene_group_ax' in axd:
    axd['gene_group_ax'].set_visible(False)
add_brackets(axd, marker_genes)

plt.savefig('dotplot_bcr_lineage_markers.pdf', bbox_inches='tight', dpi=300)
plt.show()

### scBCR Barcode integration distribution

In [ ]:
bcr_mask  = adata.obs['in_bcr']
bcr_total = int(bcr_mask.sum())

def n(name, within_bcr):
    m = adata.obs['major_celltype'] == name
    return int((m & bcr_mask).sum()) if within_bcr else int(m.sum())

lineages = ['B cells', 'T cells', 'NK cells', 'DCs']
overlap  = {lin: n(lin, True)  for lin in lineages}
total    = {lin: n(lin, False) for lin in lineages}
bcr_other = bcr_total - sum(overlap.values())

# --- summary table: per-lineage BCR breakdown ---
summary = pd.DataFrame({
    'total_cells':   {lin: total[lin]   for lin in lineages},
    'BCR+':          {lin: overlap[lin] for lin in lineages},
})
summary['BCR-'] = summary['total_cells'] - summary['BCR+']
summary['% of lineage BCR+'] = (summary['BCR+'] / summary['total_cells'] * 100).round(1)
summary['% of all BCR+']     = (summary['BCR+'] / bcr_total * 100).round(1)

summary.loc['— all 4 lineages —'] = [
    summary['total_cells'].sum(), summary['BCR+'].sum(), summary['BCR-'].sum(),
    round(summary['BCR+'].sum() / summary['total_cells'].sum() * 100, 1),
    round(summary['BCR+'].sum() / bcr_total * 100, 1),
]

print("=" * 62)
print(f"Total cells in adata:        {adata.n_obs:,}")
print(f"Total BCR+ cells:            {bcr_total:,}")
print(f"BCR+ outside B/T/NK/DC:      {bcr_other:,} "
      f"({bcr_other/bcr_total:.1%} of all BCR+)")
print("=" * 62)
print(summary.to_string())
print("=" * 62)

for lin in lineages:
    if total[lin]:
        print(f"  {lin:9s}: {overlap[lin]:>5,} / {total[lin]:>6,} carry a BCR "
              f"({overlap[lin]/total[lin]:5.1%} of {lin}; "
              f"{overlap[lin]/bcr_total:5.1%} of all BCR+)")

# --- stacked bar plot: BCR+ vs BCR- counts per lineage (absolute) ---
plot_df = summary.loc[lineages].copy()
plot_df = plot_df.sort_values('BCR+', ascending=False)

fig, ax = plt.subplots(figsize=(7, 9))

x = np.arange(len(plot_df))
bcr_pos = plot_df['BCR+'].values
bcr_neg = plot_df['BCR-'].values

ax.bar(x, bcr_pos, color='#3b598d', edgecolor='#1f3b4d', lw=1.2, label='BCR+')
ax.bar(x, bcr_neg, bottom=bcr_pos, color='#d9d9d9', edgecolor='#1f3b4d', lw=1.2, label='BCR-')

for i, lin in enumerate(plot_df.index):
    total_i = plot_df.loc[lin, 'total_cells']
    frac = plot_df.loc[lin, '% of lineage BCR+']
    ax.text(i, bcr_pos[i] / 2, f'{int(bcr_pos[i]):,}\n({frac:.1f}%)',
            ha='center', va='center', fontsize=9)
    ax.text(i, total_i + total_i * 0.02, f'n={int(total_i):,}',
            ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(plot_df.index)
ax.set_ylabel('Cell count')
ax.set_title(f'BCR+ vs BCR- cells per lineage\n'
             f'(total BCR+ = {bcr_total:,}; BCR+ not in B/T/NK/DC = {bcr_other:,})')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
#plt.savefig('bcr_celltype_barplot.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# --- scBCR cells/sequences by sample/subject + chain distribution (reuses `scbcr_df`) ---
scbcr_sample_col  = pick_first_existing(scbcr_df, ['sample_id', 'sample'])
scbcr_subject_col = pick_first_existing(scbcr_df, ['subject_id', 'subject'])
scbcr_chain_col   = pick_first_existing(scbcr_df, ['locus', 'chain', 'chain_type'])

if scbcr_sample_col and scbcr_sample_col != 'sample_id':
    scbcr_df['sample_id'] = scbcr_df[scbcr_sample_col].astype(str)
if scbcr_subject_col and scbcr_subject_col != 'subject_id':
    scbcr_df['subject_id'] = scbcr_df[scbcr_subject_col].astype(str)

scbcr_by_sample = (
    summarize_counts(scbcr_df, 'scBCR', 'sample_id', 'cell_id', 'sample_id')
    if 'sample_id' in scbcr_df.columns else None
)
scbcr_by_subject = (
    summarize_counts(scbcr_df, 'scBCR', 'subject_id', 'cell_id', 'subject_id')
    if 'subject_id' in scbcr_df.columns else None
)
scbcr_chain_by_sample = (
    summarize_chain_distribution(scbcr_df, 'sample_id', scbcr_chain_col)
    if 'sample_id' in scbcr_df.columns else None
)

show_subheader('scBCR cells/sequences by sample')
show_table(scbcr_by_sample)
plot_counts(scbcr_by_sample, 'sample_id', 'scBCR Cells and Sequences per Sample')

show_subheader('scBCR sequence chain distribution by sample')
if scbcr_chain_col:
    show_table(scbcr_chain_by_sample)
plot_chain_distribution(scbcr_chain_by_sample, f'scBCR Chain Distribution per Sample ({scbcr_chain_col})')

### scBCR isotype vs celltype

In [ ]:
merged_selected = scbcr_df[scbcr_df["locus"]=="IGH"]
merged_selected = merged_selected[merged_selected["bcell"]=="yes"]

In [ ]:
merged_selected['isotype'] = (
    merged_selected['c_call']
    .str.split(",")
    .str[0]
    .str[:4]
)

In [ ]:
merged_selected["isotype"] = merged_selected["isotype"].map(
    lambda x: "".join(x[i] for i in [0, 1, 3]) if isinstance(x, str) and len(x) > 3 else x
)

In [ ]:
# alakazam IG_COLORS, with the trailing "H" dropped to match your labels
isotype_colors = {
    "IGA": "#377EB8",
    "IGD": "#FF7F00",
    "IGE": "#E41A1C",
    "IGG": "#4DAF4A",
    "IGM": "#984EA3",
}

In [ ]:
composition = (
    merged_selected
    .groupby("cell_type", observed=True)["isotype"]
    .value_counts(normalize=False, dropna=True)
    .unstack(fill_value=0)
)

order = ["IGD", "IGM", "IGA", "IGG", "IGE"]
composition = composition.reindex(columns=order, fill_value=0)

totals = composition.sum(axis=1)
celltype_order = list(totals.sort_values(ascending=False).index)
composition_pct = composition.div(totals, axis=0) * 100

# --- plot (no N= labels) ---
fig, ax = plt.subplots(figsize=(8, 5))
composition_pct.loc[celltype_order].plot.bar(
    stacked=True,
    ax=ax,
    color=[isotype_colors[iso] for iso in composition_pct.columns],  # <-- colors
)
ax.set_ylabel("Isotype Percentage")
ax.set_xlabel("B cell subtypes")
plt.xticks(rotation=30, ha="right", fontsize=8)
plt.yticks(fontsize=11)
ax.grid(False)
ax.legend(title="Isotype", loc="upper right", bbox_to_anchor=(1.15, 1), borderaxespad=0)
plt.tight_layout()
plt.show()

# --- table of cell counts, shown after the plot ---
count_table = composition.loc[celltype_order].copy()
count_table["Total (N)"] = totals.loc[celltype_order]
count_table.index.name = "B cell subtype"
display(count_table)

count_table.to_csv("isotype_counts_by_celltype.csv")

### scBCR mutation frequency vs celltype

In [ ]:
isotype_order = ["IGD", "IGM", "IGA", "IGG", "IGE"]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(merged_selected, x="cell_type", y="mu_freq", hue= "isotype", hue_order=isotype_order,palette=isotype_colors, fill=False)
ax.grid(False)
ax.set_title("Mutation Frequency", fontsize=15)
ax.set_xlabel("B cell subtypes", fontsize=13)
ax.set_ylabel("Mutation Frequency", fontsize = 13)
ax.tick_params(axis="x", rotation=30, labelsize=12)
for label in ax.get_xticklabels():
    label.set_ha("right")
ax.tick_params(axis="y", labelsize=12)
ax.legend(
    title="Isotype",
    loc="upper right",
    bbox_to_anchor=(1.15, 1),  # move legend outside the plot
    borderaxespad=0
)
plt.tight_layout()
plt.show()

### BCR statistics

In [ ]:
adata.obs['sample_id'] = adata.obs['sample'].str.replace('_filtered_feature_bc_matrix', '', regex=False)
adata.obs['cell_id_scIntegrator'] = adata.obs.index.str.split("-").str[:2].str.join('-')

In [ ]:
print(sorted(adata.obs['sample_id'].unique()))

In [ ]:
B_cells = [
    "Memory B cells",
    "Naive B cells",
    "Plasma cells",
    "Age-associated B cells",
    "Plasmablasts",
    "Proliferative germinal center B cells",
    "Transitional B cells",
    "B cells",
    "Small pre-B cells",
    "Germinal center B cells",
    "Pro-B cells"
]

In [ ]:
adata_B = adata[adata.obs['cell_type'].isin(B_cells)]

In [ ]:
scbcr_df['cell_id_cleaned'] = scbcr_df['cell_id'].str.split('-').str[:-1].str.join('-')

In [ ]:
############# Perform merge ##############
scbcr_df['cell_id_cleaned'] = scbcr_df['cell_id'].str.split('-').str[:-1].str.join('-')

merged_df = scbcr_df.merge(
    adata_B.obs[["sample_id","cell_id_scIntegrator","cell_type"]],
    how='inner',
    left_on=['sample_id', 'cell_id_cleaned'],
    right_on=['sample_id', 'cell_id_scIntegrator'],
)

In [ ]:
merged_df["isotype"]=merged_df["c_call"].map(lambda x: x.split(",")[0][:4])
merged_df['study_name'] = "D3"
clone_size = (
    merged_df[merged_df["locus"] == "IGH"]
    .groupby(["study_name","subject_id", "sample_id","clone_id"])
    .size()
)
merged_df["mu_freq"]=merged_df["mu_freq"].astype(float)
merged_df["cdr3_length"]=merged_df["cdr3"].map(lambda x: len(x))
outdir = "/vast/palmer/pi/kleinstein/huimin/ABCD_D3/statistics"

In [ ]:
# ---- Collect every summary table in memory; written to ONE file at the end ----
LEVELS = {"STUDY": "study_name", "SUBJECT": "subject_id", "SAMPLE": "sample_id"}
results = {}   # {table_name: [DataFrame per level, ...]}

def collect(table, fn):
    """Run a summary function at STUDY / SUBJECT / SAMPLE level and store the stacked result.
    Output columns: level, group_id, <the function's own columns>."""
    parts = []
    for level, group in LEVELS.items():
        df = fn(merged_df=merged_df, group=group).reset_index()
        if group in df.columns:
            df = df.rename(columns={group: "group_id"})
        else:                                   # safety net if the index name was lost
            df = df.rename(columns={df.columns[0]: "group_id"})
        df.insert(0, "level", level)
        parts.append(df)
    results[table] = pd.concat(parts, ignore_index=True)


#### Summary table

In [ ]:
def Brief_Summary(merged_df, group):
    count_bcrs=merged_df.groupby([group])["cell_id"].nunique()
    count_igh_igk_pairs = merged_df[merged_df["locus"]=="IGK"].groupby(group)["cell_id"].nunique()
    count_igh_igl_pairs = merged_df[merged_df["locus"]=="IGL"].groupby(group)["cell_id"].nunique()
    count_igm = merged_df[merged_df["isotype"]=="IGHM"].groupby(group).size()
    count_igd = merged_df[merged_df["isotype"]=="IGHD"].groupby(group).size()
    count_iga = merged_df[merged_df["isotype"]=="IGHA"].groupby(group).size()
    count_igg = merged_df[merged_df["isotype"]=="IGHG"].groupby(group).size()
    count_ige = merged_df[merged_df["isotype"]=="IGHE"].groupby(group).size()
    clonesize_mean = clone_size.groupby(group).mean()
    clonesize_median = clone_size.groupby(group).median()
    clonesize_p05=clone_size.groupby(group).quantile(0.05)
    clonesize_p25=clone_size.groupby(group).quantile(0.25)
    clonesize_p75=clone_size.groupby(group).quantile(0.75)
    clonesize_p95=clone_size.groupby(group).quantile(0.95)
    largest_clonesize = clone_size.groupby(group).max()
    OUTPUT_SUMMARY = pd.DataFrame({
    "count_bcrs": count_bcrs,
    "count_igh_igk_pairs": count_igh_igk_pairs,
    "count_igh_igl_pairs": count_igh_igl_pairs,
    "count_igm": count_igm,
    "count_igd": count_igd,   
    "count_iga": count_iga,
    "count_igg": count_igg,
    "count_ige": count_ige,
    "clonesize_mean": clonesize_mean,
    "clonesize_median": clonesize_median,
    "clonesize_p05": clonesize_p05,
    "clonesize_p25": clonesize_p25,
    "clonesize_p75": clonesize_p75,
    "clonesize_p95": clonesize_p95,
    "largest_clonesize": largest_clonesize
    })
    return OUTPUT_SUMMARY

In [ ]:
collect("SUMMARY", Brief_Summary)

#### Chain Summary

In [ ]:
def Chain_Summary(merged_df, group):
    count_chain=merged_df.groupby([group, "locus"]).size()
    mufreq_mean = merged_df.groupby([group, "locus"])["mu_freq"].mean()
    mufreq_median = merged_df.groupby([group, "locus"])["mu_freq"].median()
    mufreq_p05=merged_df.groupby([group, "locus"])["mu_freq"].quantile(0.05)
    mufreq_p25=merged_df.groupby([group, "locus"])["mu_freq"].quantile(0.25)
    mufreq_p75=merged_df.groupby([group, "locus"])["mu_freq"].quantile(0.75)
    mufreq_p95=merged_df.groupby([group, "locus"])["mu_freq"].quantile(0.95)
    cdr3_mean = merged_df.groupby([group, "locus"])["cdr3_length"].mean()
    cdr3_median = merged_df.groupby([group, "locus"])["cdr3_length"].median()
    cdr3_p05=merged_df.groupby([group, "locus"])["cdr3_length"].quantile(0.05)
    cdr3_p25=merged_df.groupby([group, "locus"])["cdr3_length"].quantile(0.25)
    cdr3_p75=merged_df.groupby([group, "locus"])["cdr3_length"].quantile(0.75)
    cdr3_p95=merged_df.groupby([group, "locus"])["cdr3_length"].quantile(0.95)
    OUTPUT_SUMMARY = pd.DataFrame({
    "count_chain": count_chain,
    "mufreq_mean": mufreq_mean,
    "mufreq_median": mufreq_median,
    "mufreq_p05": mufreq_p05,
    "mufreq_p25": mufreq_p25,
    "mufreq_p75": mufreq_p75,
    "mufreq_p95": mufreq_p95,
    "cdr3_mean": cdr3_mean,
    "cdr3_median": cdr3_median,
    "cdr3_p05": cdr3_p05,
    "cdr3_p25": cdr3_p25,
    "cdr3_p75": cdr3_p75,
    "cdr3_p95": cdr3_p95
    })
    return OUTPUT_SUMMARY

In [ ]:
collect("CHAIN_SUMMARY", Chain_Summary)

### Celltype Summary

In [ ]:
def CELLTYPE_Summary(merged_df, group):
    count_bcrs=merged_df.groupby([group,"cell_type"], observed=True)["cell_id"].nunique()
    count_igm = merged_df[merged_df["isotype"]=="IGHM"].groupby([group,"cell_type"], observed=True).size()
    count_igd = merged_df[merged_df["isotype"]=="IGHD"].groupby([group,"cell_type"], observed=True).size()
    count_iga = merged_df[merged_df["isotype"]=="IGHA"].groupby([group,"cell_type"], observed=True).size()
    count_igg = merged_df[merged_df["isotype"]=="IGHG"].groupby([group,"cell_type"], observed=True).size()
    count_ige = merged_df[merged_df["isotype"]=="IGHE"].groupby([group,"cell_type"], observed=True).size()
    CELLTYPE_SUMMARY = pd.DataFrame({
    "count_bcrs": count_bcrs,
    "count_igm": count_igm,
    "count_igd": count_igd,   
    "count_iga": count_iga,
    "count_igg": count_igg,
    "count_ige": count_ige
    })
    return CELLTYPE_SUMMARY

In [ ]:
collect("CELLTYPE_SUMMARY", CELLTYPE_Summary)

#### Chain and Celltype Summary

In [ ]:
def Chain_CELLTYPE_Summary(merged_df, group):
    count_chain=merged_df.groupby([group, "locus","cell_type"], observed=True).size()
    mufreq_mean = merged_df.groupby([group, "locus","cell_type"], observed=True)["mu_freq"].mean()
    mufreq_median = merged_df.groupby([group, "locus","cell_type"], observed=True)["mu_freq"].median()
    mufreq_p05=merged_df.groupby([group, "locus","cell_type"], observed=True)["mu_freq"].quantile(0.05)
    mufreq_p25=merged_df.groupby([group, "locus","cell_type"], observed=True)["mu_freq"].quantile(0.25)
    mufreq_p75=merged_df.groupby([group, "locus","cell_type"], observed=True)["mu_freq"].quantile(0.75)
    mufreq_p95=merged_df.groupby([group, "locus","cell_type"], observed=True)["mu_freq"].quantile(0.95)
    cdr3_mean = merged_df.groupby([group, "locus","cell_type"], observed=True)["cdr3_length"].mean()
    cdr3_median = merged_df.groupby([group, "locus","cell_type"], observed=True)["cdr3_length"].median()
    cdr3_p05=merged_df.groupby([group, "locus","cell_type"], observed=True)["cdr3_length"].quantile(0.05)
    cdr3_p25=merged_df.groupby([group, "locus","cell_type"], observed=True)["cdr3_length"].quantile(0.25)
    cdr3_p75=merged_df.groupby([group, "locus","cell_type"], observed=True)["cdr3_length"].quantile(0.75)
    cdr3_p95=merged_df.groupby([group, "locus","cell_type"], observed=True)["cdr3_length"].quantile(0.95)
    OUTPUT_SUMMARY = pd.DataFrame({
    "count_chain": count_chain,
    "mufreq_mean": mufreq_mean,
    "mufreq_median": mufreq_median,
    "mufreq_p05": mufreq_p05,
    "mufreq_p25": mufreq_p25,
    "mufreq_p75": mufreq_p75,
    "mufreq_p95": mufreq_p95,
    "cdr3_mean": cdr3_mean,
    "cdr3_median": cdr3_median,
    "cdr3_p05": cdr3_p05,
    "cdr3_p25": cdr3_p25,
    "cdr3_p75": cdr3_p75,
    "cdr3_p95": cdr3_p95
    })
    return OUTPUT_SUMMARY

In [ ]:
collect("CHAIN_CELLTYPE_SUMMARY", Chain_CELLTYPE_Summary)

#### VJ Allele Summary

In [ ]:
merged_df["v_allele"]=merged_df["v_call"].map(lambda x: x.split(",")[0])
merged_df["j_allele"]=merged_df["j_call"].map(lambda x: x.split(",")[0])

In [ ]:
def VJ_ALLELE(merged_df, group):
    count_allele_v=merged_df.groupby([group, "v_allele"], observed=True).size()
    count_allele_j=merged_df.groupby([group, "j_allele"], observed=True).size()
    OUTPUT_SUMMARY_v = pd.DataFrame({
    "count_allele_v": count_allele_v
    })
    OUTPUT_SUMMARY_v = OUTPUT_SUMMARY_v.reset_index(level="v_allele")
    OUTPUT_SUMMARY_v.rename(columns={"v_allele": "vj_allele", "count_allele_v" : "count_vj_allele"}, inplace=True)
    OUTPUT_SUMMARY_j = pd.DataFrame({
    "count_allele_j": count_allele_j
    })
    OUTPUT_SUMMARY_j = OUTPUT_SUMMARY_j.reset_index(level="j_allele")
    OUTPUT_SUMMARY_j.rename(columns={"j_allele": "vj_allele", "count_allele_j" : "count_vj_allele"}, inplace=True)
    OUTPUT_SUMMARY_vj =  pd.concat([OUTPUT_SUMMARY_v, OUTPUT_SUMMARY_j])
    return OUTPUT_SUMMARY_vj

In [ ]:
collect("VJ_ALLELE", VJ_ALLELE)

#### VJ Allele_Celltype

In [ ]:
def VJ_ALLELE_CELLTYPE(merged_df, group):
    count_allele_v=merged_df.groupby([group, "cell_type", "v_allele"], observed=True).size()
    count_allele_j=merged_df.groupby([group, "cell_type", "j_allele"], observed=True).size()
    OUTPUT_SUMMARY_v = pd.DataFrame({
    "count_allele_v": count_allele_v
    })
    OUTPUT_SUMMARY_v = OUTPUT_SUMMARY_v.reset_index(level="v_allele")
    OUTPUT_SUMMARY_v.rename(columns={"v_allele": "vj_allele", "count_allele_v" : "count_vj_allele"}, inplace=True)
    OUTPUT_SUMMARY_j = pd.DataFrame({
    "count_allele_j": count_allele_j
    })
    OUTPUT_SUMMARY_j = OUTPUT_SUMMARY_j.reset_index(level="j_allele")
    OUTPUT_SUMMARY_j.rename(columns={"j_allele": "vj_allele", "count_allele_j" : "count_vj_allele"}, inplace=True)
    OUTPUT_SUMMARY_vj =  pd.concat([OUTPUT_SUMMARY_v, OUTPUT_SUMMARY_j])
    return OUTPUT_SUMMARY_vj

In [ ]:
collect("VJ_ALLELE_CELLTYPE", VJ_ALLELE_CELLTYPE)

#### Write Tables

In [ ]:
############ WRITE ALL TABLES TO ONE FILE ###########
OUTPUT_FORMAT = "sqlite"      # "sqlite" (default) or "xlsx"
os.makedirs(outdir, exist_ok=True)

if OUTPUT_FORMAT == "sqlite":
    import sqlite3
    out_file = os.path.join(outdir, "D3_statistics.db")
    with sqlite3.connect(out_file) as con:
        for name, df in results.items():
            df.to_sql(name, con, if_exists="replace", index=False)
elif OUTPUT_FORMAT == "xlsx":
    out_file = os.path.join(outdir, "D3_statistics.xlsx")
    with pd.ExcelWriter(out_file) as xl:
        for name, df in results.items():
            df.to_excel(xl, sheet_name=name, index=False)
else:
    raise ValueError("OUTPUT_FORMAT must be 'sqlite' or 'xlsx'")

print("Wrote", out_file)
for name, df in results.items():
    print(f"  {name}: {df.shape[0]} rows x {df.shape[1]} cols")

# ---- How to read it back in Python ----
# SQLite : pd.read_sql("SELECT * FROM CHAIN_SUMMARY WHERE level='SAMPLE'", sqlite3.connect(out_file))
# Excel  : tables = pd.read_excel(out_file, sheet_name=None)   # dict of DataFrames